# Molecular Classification - Simplified
Load molecules, apply binning (TPSA, LogP, MW), calculate LogD and CNS MPO, include BBB tags, export CSV

In [1]:
import pandas as pd
import numpy as np
import subprocess
import sys

# Install RDKit for chemistry calculations
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"])
print("✓ RDKit installed")

✓ RDKit installed


In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Crippen
import json

# Load raw data
data_path = "../../mtoralzml/data/padel_loop_results_BBB.csv"
raw_df = pd.read_csv(data_path)
print(f"✓ Loaded {len(raw_df)} molecules from {data_path}")
print(f"  Columns: {list(raw_df.columns[:15])}...")

✓ Loaded 9584 molecules from ../../mtoralzml/data/padel_loop_results_BBB.csv
  Columns: ['Original_Name', 'Original_SMILES', 'nAcid', 'ALogP', 'ALogp2', 'AMR', 'apol', 'naAromAtom', 'nAromBond', 'nAtom', 'nHeavyAtom', 'nH', 'nB', 'nC', 'nN']...


In [3]:
# ============================================
# BINNING FUNCTIONS (Numeric buckets)
# ============================================

def get_tpsa_bin(tpsa):
    """Bin TPSA into numeric buckets"""
    if pd.isna(tpsa):
        return None
    if tpsa <= 60:
        return 'tpsa_le_60'
    elif tpsa <= 90:
        return 'tpsa_60_90'
    elif tpsa <= 140:
        return 'tpsa_90_140'
    else:
        return 'tpsa_gt_140'

def get_logp_bin(logp):
    """Bin LogP into numeric buckets"""
    if pd.isna(logp):
        return None
    if logp < 1:
        return 'logp_lt_1'
    elif logp < 3:
        return 'logp_1_3'
    elif logp <= 5:
        return 'logp_3_5'
    else:
        return 'logp_gt_5'

def get_mw_bin(mw):
    """Bin MW into numeric buckets"""
    if pd.isna(mw):
        return None
    if mw < 300:
        return 'mw_lt_300'
    elif mw < 450:
        return 'mw_300_450'
    elif mw <= 500:
        return 'mw_450_500'
    else:
        return 'mw_gt_500'

print("✓ Binning functions defined")

✓ Binning functions defined


In [ ]:
# ============================================
# LOGD AND CNS MPO CALCULATION
# ============================================

def calculate_logd(smiles, ph=7.4):
    """Calculate LogD at physiological pH using RDKit Crippen method"""
    if pd.isna(smiles):
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        # Simplified: using LogP as proxy for LogD
        logd = Crippen.MolLogP(mol)
        return round(logd, 2)
    except:
        return None

def calculate_cns_mpo(smiles, logp, mw, tpsa):
    """Calculate CNS MPO score (0-6 scale) for drug CNS suitability"""
    if pd.isna(smiles):
        return None
    
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        
        score = 0.0
        
        # LogP: ideal 1-2.5 (1 point)
        if pd.notna(logp):
            if 1 <= logp <= 2.5:
                score += 1.0
            elif 0.5 <= logp < 1 or 2.5 < logp <= 3:
                score += 0.5
        
        # MW: ideal 160-360 (1 point)
        if pd.notna(mw):
            if 160 <= mw <= 360:
                score += 1.0
            elif 130 <= mw < 160 or 360 < mw <= 400:
                score += 0.5
        
        # TPSA: ideal 40-90 (1 point)
        if pd.notna(tpsa):
            if 40 <= tpsa <= 90:
                score += 1.0
            elif 30 <= tpsa < 40 or 90 < tpsa <= 110:
                score += 0.5
        
        # HBD: ideal 0-3 (1 point)
        hbd = Descriptors.NumHDonors(mol)
        if 0 <= hbd <= 3:
            score += 1.0
        elif hbd == 4:
            score += 0.5
        
        # Rotatable bonds: ideal ≤5 (1 point)
        rotatable = Descriptors.NumRotatableBonds(mol)
        if rotatable <= 5:
            score += 1.0
        elif rotatable == 6:
            score += 0.5
        
        # Aromatic rings: ideal ≤2 (1 point)
        aromatic_rings = Descriptors.NumAromaticRings(mol)
        if aromatic_rings <= 2:
            score += 1.0
        elif aromatic_rings == 3:
            score += 0.5
        
        return round(score, 1)
    except:
        return None

print("✓ LogD and CNS MPO functions defined")

✓ LogD and CNS MPO functions defined


In [5]:
# ============================================
# PREPARE CLASSIFICATION DATASET
# ============================================

df = pd.DataFrame()

# Basic identifiers
df['id'] = range(len(raw_df))
df['name'] = raw_df['Original_Name'].fillna('')
df['smiles'] = raw_df['Original_SMILES'].fillna('')

# Physicochemical properties
df['tpsa'] = raw_df['TPSA']
df['logp'] = raw_df['CrippenLogP']
df['mw'] = raw_df['MW']
df['hbd'] = raw_df['nHBDon_Lipinski']
df['hba'] = raw_df['nHBAcc_Lipinski']
df['rotatable_bonds'] = raw_df['nRotB']
df['ring_count'] = raw_df['nRing']
df['molar_refractivity'] = raw_df['AMR']

# BBB tag (include for all molecules)
if 'BBB' in raw_df.columns:
    df['bbb_tag'] = raw_df['BBB'].fillna('')
else:
    df['bbb_tag'] = ''

print(f"✓ Dataset prepared: {df.shape}")
print(f"  Columns: {list(df.columns)}")

✓ Dataset prepared: (9584, 12)
  Columns: ['id', 'name', 'smiles', 'tpsa', 'logp', 'mw', 'hbd', 'hba', 'rotatable_bonds', 'ring_count', 'molar_refractivity', 'bbb_tag']


In [6]:
df

,id,name,smiles,tpsa,logp,mw,hbd,hba,rotatable_bonds,ring_count,molar_refractivity,bbb_tag
0,0,sulphasalazine,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,NaN,4.08157,398.068491,3,9,6.0,3.0,20.9255,0
1,1,moxalactam,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,302.211486,-1.36323,520.101247,4,15,10.0,4.0,84.1596,0
2,2,clioquinol,Oc1c(I)cc(Cl)c2cccnc12,79.338804,3.56899,304.910439,1,2,0.0,2.0,22.1264,0
3,3,bbcpd11 (cimetidine analog) (y-g13),CCNC(=NCCSCc1ncccc1Br)NC#N,NaN,2.23388,341.030979,2,5,8.0,1.0,58.1882,0
4,4,schembl614298,CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...,NaN,-1.31031,461.168581,5,10,3.0,6.0,88.3588,0
...,...,...,...,...,...,...,...,...,...,...,...,...
9579,9579,licostinel,C1=C(Cl)C(=C(C2=C1NC(=O)C(N2)=O)[N+](=O)[O-])Cl,180.400362,1.65120,274.950061,2,7,1.0,2.0,57.1443,1
9580,9580,ademetionine(adenosyl-methionine),[C@H]3([N]2C1=C(C(=NC=N1)N)N=C2)[C@@H]([C@@H](...,NaN,-3.13810,398.137239,6,11,7.0,3.0,84.4263,1
9581,9581,mesocarb,[O+]1=N[N](C=C1[N-]C(NC2=CC=CC=C2)=O)C(CC3=CC=...,NaN,5.72957,322.142976,1,6,7.0,3.0,98.9391,1
9582,9582,tofisoline,C1=C(OC)C(=CC2=C1C(=[N+](C(=C2CC)C)[NH-])C3=CC...,NaN,5.48942,382.189257,1,6,6.0,3.0,113.2376,1


In [7]:
# ============================================
# APPLY BINNING
# ============================================

print("Calculating bins...")
df['tpsa_bin'] = df['tpsa'].apply(get_tpsa_bin)
df['logp_bin'] = df['logp'].apply(get_logp_bin)
df['mw_bin'] = df['mw'].apply(get_mw_bin)
print("✓ Binning complete")

Calculating bins...
✓ Binning complete


In [8]:
# ============================================
# CALCULATE LOGD AND CNS MPO
# ============================================

print("Calculating LogD (this may take a moment)...")
df['logd'] = df['smiles'].apply(calculate_logd)
print(f"  ✓ LogD calculated ({df['logd'].notna().sum()}/{len(df)} valid)")

print("Calculating CNS MPO score (this may take a moment)...")
df['cns_mpo'] = df.apply(
    lambda row: calculate_cns_mpo(row['smiles'], row['logp'], row['mw'], row['tpsa']),
    axis=1
)
print(f"  ✓ CNS MPO calculated ({df['cns_mpo'].notna().sum()}/{len(df)} valid)")

Calculating LogD (this may take a moment)...


[20:58:15] Explicit valence for atom # 10 C, 4, is greater than permitted
[20:58:16] Explicit valence for atom # 10 C, 4, is greater than permitted
[20:58:16] Explicit valence for atom # 1 N, 4, is greater than permitted
[20:58:16] WARNING: not removing hydrogen atom without neighbors
[20:58:16] Explicit valence for atom # 6 N, 4, is greater than permitted
[20:58:16] WARNING: not removing hydrogen atom without neighbors
[20:58:16] WARNING: not removing hydrogen atom without neighbors
[20:58:16] WARNING: not removing hydrogen atom without neighbors
[20:58:16] WARNING: not removing hydrogen atom without neighbors
[20:58:16] WARNING: not removing hydrogen atom without neighbors
[20:58:16] WARNING: not removing hydrogen atom without neighbors
[20:58:16] Explicit valence for atom # 6 N, 4, is greater than permitted
[20:58:16] WARNING: not removing hydrogen atom without neighbors
[20:58:16] WARNING: not removing hydrogen atom without neighbors
[20:58:16] WARNING: not removing hydrogen atom w

  ✓ LogD calculated (9571/9584 valid)
Calculating CNS MPO score (this may take a moment)...


[20:58:18] Explicit valence for atom # 10 C, 4, is greater than permitted
[20:58:19] Explicit valence for atom # 10 C, 4, is greater than permitted
[20:58:19] Explicit valence for atom # 1 N, 4, is greater than permitted
[20:58:19] WARNING: not removing hydrogen atom without neighbors
[20:58:19] Explicit valence for atom # 6 N, 4, is greater than permitted
[20:58:19] WARNING: not removing hydrogen atom without neighbors
[20:58:19] WARNING: not removing hydrogen atom without neighbors
[20:58:19] WARNING: not removing hydrogen atom without neighbors
[20:58:19] WARNING: not removing hydrogen atom without neighbors
[20:58:19] WARNING: not removing hydrogen atom without neighbors
[20:58:19] WARNING: not removing hydrogen atom without neighbors
[20:58:19] Explicit valence for atom # 6 N, 4, is greater than permitted
[20:58:19] WARNING: not removing hydrogen atom without neighbors
[20:58:19] WARNING: not removing hydrogen atom without neighbors
[20:58:19] WARNING: not removing hydrogen atom w

  ✓ CNS MPO calculated (9571/9584 valid)


[20:58:20] WARNING: not removing hydrogen atom without neighbors
[20:58:20] WARNING: not removing hydrogen atom without neighbors
[20:58:20] WARNING: not removing hydrogen atom without neighbors
[20:58:20] WARNING: not removing hydrogen atom without neighbors
[20:58:20] WARNING: not removing hydrogen atom without neighbors
[20:58:20] WARNING: not removing hydrogen atom without neighbors
[20:58:20] WARNING: not removing hydrogen atom without neighbors


In [9]:
# ============================================
# BUILD FINAL CLASSIFIED DATASET
# ============================================

# Select columns for output
classified = df[[
    'id',
    'name',
    'smiles',
    'tpsa',
    'logp',
    'mw',
    'hbd',
    'hba',
    'rotatable_bonds',
    'ring_count',
    'molar_refractivity',
    'tpsa_bin',
    'logp_bin',
    'mw_bin',
    'logd',
    'cns_mpo',
    'bbb_tag'
]].copy()

print(f"✓ Final dataset: {classified.shape}")
print(f"\nColumns: {list(classified.columns)}")
print(f"\nFirst 5 rows:")
classified.head()

✓ Final dataset: (9584, 17)

Columns: ['id', 'name', 'smiles', 'tpsa', 'logp', 'mw', 'hbd', 'hba', 'rotatable_bonds', 'ring_count', 'molar_refractivity', 'tpsa_bin', 'logp_bin', 'mw_bin', 'logd', 'cns_mpo', 'bbb_tag']

First 5 rows:


,id,name,smiles,tpsa,logp,mw,hbd,hba,rotatable_bonds,ring_count,molar_refractivity,tpsa_bin,logp_bin,mw_bin,logd,cns_mpo,bbb_tag
0,0,sulphasalazine,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,NaN,4.08157,398.068491,3,9,6.0,3.0,20.9255,None,logp_3_5,mw_300_450,3.70,2.5,0
1,1,moxalactam,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,302.211486,-1.36323,520.101247,4,15,10.0,4.0,84.1596,tpsa_gt_140,logp_lt_1,mw_gt_500,-1.13,1.5,0
2,2,clioquinol,Oc1c(I)cc(Cl)c2cccnc12,79.338804,3.56899,304.910439,1,2,0.0,2.0,22.1264,tpsa_60_90,logp_3_5,mw_300_450,3.20,5.0,0
3,3,bbcpd11 (cimetidine analog) (y-g13),CCNC(=NCCSCc1ncccc1Br)NC#N,NaN,2.23388,341.030979,2,5,8.0,1.0,58.1882,None,logp_1_3,mw_300_450,2.11,4.5,0
4,4,schembl614298,CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...,NaN,-1.31031,461.168581,5,10,3.0,6.0,88.3588,None,logp_lt_1,mw_450_500,-1.24,2.0,0


In [10]:
# ============================================
# EXPORT TO CSV
# ============================================

output_file = 'classified_molecules.csv'
classified.to_csv(output_file, index=False)

print(f"\n✓ Classification complete!")
print(f"\nExported to: {output_file}")
print(f"\nSummary:")
print(f"  Total molecules: {len(classified)}")
print(f"  Columns: {len(classified.columns)}")
print(f"  File size: {pd.io.common.get_filepath_or_buffer(output_file)[0] if False else 'Generated'}")
print(f"\nBinning distribution:")
print(f"  TPSA bins: {classified['tpsa_bin'].value_counts().to_dict()}")
print(f"  LogP bins: {classified['logp_bin'].value_counts().to_dict()}")
print(f"  MW bins: {classified['mw_bin'].value_counts().to_dict()}")
print(f"\nCNS MPO score distribution:")
print(classified['cns_mpo'].describe())


✓ Classification complete!

Exported to: classified_molecules.csv

Summary:
  Total molecules: 9584
  Columns: 17
  File size: Generated

Binning distribution:
  TPSA bins: {'tpsa_gt_140': 2615, 'tpsa_le_60': 1829, 'tpsa_90_140': 1575, 'tpsa_60_90': 979}
  LogP bins: {'logp_1_3': 3481, 'logp_3_5': 3310, 'logp_lt_1': 2058, 'logp_gt_5': 735}
  MW bins: {'mw_300_450': 4325, 'mw_lt_300': 3005, 'mw_gt_500': 1367, 'mw_450_500': 887}

CNS MPO score distribution:
count    9571.000000
mean        3.636193
std         1.258045
min         0.000000
25%         3.000000
50%         4.000000
75%         4.500000
max         6.000000
Name: cns_mpo, dtype: float64


In [ ]:
c_json = "classified_molecules.json"


In [11]:
# ============================================
# LOAD PROFILE JSON AND ADD TO DATAFRAME
# ============================================

# Load the JSON data from previous classification
with open('classified_molecules.json', 'r') as f:
    json_data = json.load(f)

print(f"✓ Loaded {len(json_data)} profile JSONs")

# Create a dictionary mapping row index to profile_json string
profile_map = {}
for idx, entry in enumerate(json_data):
    # Create minimal JSON profile from the entry
    profile = {
        "name": entry.get('name', ''),
        "tpsa": entry.get('tpsa'),
        "logp": entry.get('logp'),
        "mw": entry.get('mw'),
        "hbd": entry.get('hbd'),
        "hba": entry.get('hba'),
        "rotatable_bonds": entry.get('rotatable_bonds'),
        "ring_count": entry.get('ring_count'),
        "molar_refractivity": entry.get('molar_refractivity'),
        "bbb_tag": entry.get('bbb_tag', ''),
        "tpsa_bin": entry.get('tpsa_bin'),
        "logp_bin": entry.get('logp_bin'),
        "mw_bin": entry.get('mw_bin'),
        "logd": entry.get('logd'),
        "cns_mpo": entry.get('cns_mpo')
    }
    profile_map[idx] = json.dumps(profile)

# Add profile_json column to classified dataframe
classified['profile_json'] = classified.index.map(profile_map)

print(f"✓ Added profile_json column to {len(classified)} rows")

✓ Loaded 9584 profile JSONs
✓ Added profile_json column to 9584 rows


In [12]:
# ============================================
# RE-EXPORT CSV WITH PROFILE_JSON COLUMN
# ============================================

# Create final dataset with all columns including profile_json
final_classified = classified.copy()

# Reorder columns to put profile_json at the end
cols = list(final_classified.columns)
if 'profile_json' in cols:
    cols.remove('profile_json')
cols.append('profile_json')
final_classified = final_classified[cols]

# Export to CSV
output_file = 'classified_molecules.csv'
final_classified.to_csv(output_file, index=False)

print(f"\n✓ Re-exported to: {output_file}")
print(f"  Total rows: {len(final_classified)}")
print(f"  Total columns: {len(final_classified.columns)}")
print(f"  Columns: {list(final_classified.columns)}")
print(f"\n✓ CSV now includes profile_json column with all molecule metadata")


✓ Re-exported to: classified_molecules.csv
  Total rows: 9584
  Total columns: 18
  Columns: ['id', 'name', 'smiles', 'tpsa', 'logp', 'mw', 'hbd', 'hba', 'rotatable_bonds', 'ring_count', 'molar_refractivity', 'tpsa_bin', 'logp_bin', 'mw_bin', 'logd', 'cns_mpo', 'bbb_tag', 'profile_json']

✓ CSV now includes profile_json column with all molecule metadata


In [13]:
df1 = pd.read_csv('classified_molecules.csv')
df1

,id,name,smiles,tpsa,logp,mw,hbd,hba,rotatable_bonds,ring_count,molar_refractivity,tpsa_bin,logp_bin,mw_bin,logd,cns_mpo,bbb_tag,profile_json
0,0,sulphasalazine,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,NaN,4.08157,398.068491,3,9,6.0,3.0,20.9255,NaN,logp_3_5,mw_300_450,3.70,2.5,0,"{""name"": ""sulphasalazine"", ""tpsa"": null, ""logp..."
1,1,moxalactam,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,302.211486,-1.36323,520.101247,4,15,10.0,4.0,84.1596,tpsa_gt_140,logp_lt_1,mw_gt_500,-1.13,1.5,0,"{""name"": ""moxalactam"", ""tpsa"": 302.2114857918,..."
2,2,clioquinol,Oc1c(I)cc(Cl)c2cccnc12,79.338804,3.56899,304.910439,1,2,0.0,2.0,22.1264,tpsa_60_90,logp_3_5,mw_300_450,3.20,5.0,0,"{""name"": ""clioquinol"", ""tpsa"": 79.3388035276, ..."
3,3,bbcpd11 (cimetidine analog) (y-g13),CCNC(=NCCSCc1ncccc1Br)NC#N,NaN,2.23388,341.030979,2,5,8.0,1.0,58.1882,NaN,logp_1_3,mw_300_450,2.11,4.5,0,"{""name"": ""bbcpd11 (cimetidine analog) (y-g13)""..."
4,4,schembl614298,CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...,NaN,-1.31031,461.168581,5,10,3.0,6.0,88.3588,NaN,logp_lt_1,mw_450_500,-1.24,2.0,0,"{""name"": ""schembl614298"", ""tpsa"": null, ""logp""..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9579,9579,licostinel,C1=C(Cl)C(=C(C2=C1NC(=O)C(N2)=O)[N+](=O)[O-])Cl,180.400362,1.65120,274.950061,2,7,1.0,2.0,57.1443,tpsa_gt_140,logp_1_3,mw_lt_300,1.43,5.0,1,"{""name"": ""licostinel"", ""tpsa"": 180.4003621705,..."
9580,9580,ademetionine(adenosyl-methionine),[C@H]3([N]2C1=C(C(=NC=N1)N)N=C2)[C@@H]([C@@H](...,NaN,-3.13810,398.137239,6,11,7.0,3.0,84.4263,NaN,logp_lt_1,mw_300_450,-3.26,2.0,1,"{""name"": ""ademetionine(adenosyl-methionine)"", ..."
9581,9581,mesocarb,[O+]1=N[N](C=C1[N-]C(NC2=CC=CC=C2)=O)C(CC3=CC=...,NaN,5.72957,322.142976,1,6,7.0,3.0,98.9391,NaN,logp_gt_5,mw_300_450,4.80,3.5,1,"{""name"": ""mesocarb"", ""tpsa"": null, ""logp"": 5.7..."
9582,9582,tofisoline,C1=C(OC)C(=CC2=C1C(=[N+](C(=C2CC)C)[NH-])C3=CC...,NaN,5.48942,382.189257,1,6,6.0,3.0,113.2376,NaN,logp_gt_5,mw_300_450,4.51,2.5,1,"{""name"": ""tofisoline"", ""tpsa"": null, ""logp"": 5..."


# Merge Missing Columns from Old CSV
Restore boolean columns (lipinski_pass, veber_pass, egan_pass, ghose_pass, pains_flag, aromatic, heterocycle_present, peptide_like, lipid_like) from old CSV

In [14]:
# ============================================
# LOAD OLD AND NEW CSVs FOR MERGING
# ============================================

# Load old CSV with boolean columns
old_csv_path = 'classified_molecules_old.csv'
old_df = pd.read_csv(old_csv_path)

print(f"✓ Old CSV loaded: {old_df.shape}")
print(f"  Old columns: {list(old_df.columns)}")

# Load new CSV (the current final_classified in memory or reload)
new_df = final_classified.copy()
print(f"\n✓ New CSV loaded: {new_df.shape}")
print(f"  New columns: {list(new_df.columns)}")

# Identify missing boolean columns
missing_cols = ['heterocycle_present', 'peptide_like', 'lipid_like', 'aromatic', 
                'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass', 'pains_flag']
print(f"\nMissing columns to restore: {missing_cols}")

✓ Old CSV loaded: (9584, 24)
  Old columns: ['id', 'name', 'smiles', 'tpsa', 'logp', 'mw', 'hbd', 'hba', 'rotatable_bonds', 'ring_count', 'heterocycle_present', 'molar_refractivity', 'peptide_like', 'lipid_like', 'polarity_bin', 'lipophilicity_bin', 'size_bin', 'aromatic', 'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass', 'pains_flag', 'profile_json']

✓ New CSV loaded: (9584, 18)
  New columns: ['id', 'name', 'smiles', 'tpsa', 'logp', 'mw', 'hbd', 'hba', 'rotatable_bonds', 'ring_count', 'molar_refractivity', 'tpsa_bin', 'logp_bin', 'mw_bin', 'logd', 'cns_mpo', 'bbb_tag', 'profile_json']

Missing columns to restore: ['heterocycle_present', 'peptide_like', 'lipid_like', 'aromatic', 'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass', 'pains_flag']


In [15]:
# ============================================
# MERGE MISSING COLUMNS FROM OLD CSV
# ============================================

# Merge on 'id' to preserve row order
merged_df = new_df.copy()

# Add missing boolean columns by matching on id
for col in missing_cols:
    if col in old_df.columns:
        # Create a mapping from id to column value
        col_mapping = dict(zip(old_df['id'].astype(str), old_df[col]))
        
        # Map values to new dataframe
        merged_df[col] = merged_df['id'].astype(str).map(col_mapping)
        
        # Count how many values were filled
        filled_count = merged_df[col].notna().sum()
        print(f"  ✓ {col}: {filled_count} values restored")

print(f"\n✓ Merged dataset: {merged_df.shape}")
print(f"  New columns: {list(merged_df.columns)}")

  ✓ heterocycle_present: 9584 values restored
  ✓ peptide_like: 9571 values restored
  ✓ lipid_like: 9571 values restored
  ✓ aromatic: 9571 values restored
  ✓ lipinski_pass: 9584 values restored
  ✓ veber_pass: 6998 values restored
  ✓ egan_pass: 6998 values restored
  ✓ ghose_pass: 9520 values restored
  ✓ pains_flag: 9571 values restored

✓ Merged dataset: (9584, 27)
  New columns: ['id', 'name', 'smiles', 'tpsa', 'logp', 'mw', 'hbd', 'hba', 'rotatable_bonds', 'ring_count', 'molar_refractivity', 'tpsa_bin', 'logp_bin', 'mw_bin', 'logd', 'cns_mpo', 'bbb_tag', 'profile_json', 'heterocycle_present', 'peptide_like', 'lipid_like', 'aromatic', 'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass', 'pains_flag']


In [16]:
# ============================================
# REORDER COLUMNS AND EXPORT FINAL CSV
# ============================================

# Define final column order: basic info -> physicochemical -> binning -> drug rules -> brain properties -> profile
final_columns = [
    'id', 'name', 'smiles',
    'tpsa', 'logp', 'mw', 'hbd', 'hba', 'rotatable_bonds', 'ring_count', 'molar_refractivity',
    'heterocycle_present', 'peptide_like', 'lipid_like', 'aromatic',
    'tpsa_bin', 'logp_bin', 'mw_bin',
    'logd', 'cns_mpo', 'bbb_tag',
    'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass', 'pains_flag',
    'profile_json'
]

# Reorder columns
final_merged_df = merged_df[final_columns].copy()

# Export to CSV
output_file = 'classified_molecules.csv'
final_merged_df.to_csv(output_file, index=False)

print(f"\n✓ Final merged CSV exported: {output_file}")
print(f"  Rows: {len(final_merged_df)}")
print(f"  Columns: {len(final_merged_df.columns)}")
print(f"  Column order: {list(final_merged_df.columns)}")



✓ Final merged CSV exported: classified_molecules.csv
  Rows: 9584
  Columns: 27
  Column order: ['id', 'name', 'smiles', 'tpsa', 'logp', 'mw', 'hbd', 'hba', 'rotatable_bonds', 'ring_count', 'molar_refractivity', 'heterocycle_present', 'peptide_like', 'lipid_like', 'aromatic', 'tpsa_bin', 'logp_bin', 'mw_bin', 'logd', 'cns_mpo', 'bbb_tag', 'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass', 'pains_flag', 'profile_json']


In [17]:
# ============================================
# VERIFICATION: SAMPLE AND SUMMARY
# ============================================

print("✓ Sample of merged data:")
display(final_merged_df[['id', 'name', 'aromatic', 'heterocycle_present', 
                          'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass', 'pains_flag']].head(10))

print("\n✓ Data type check:")
print(final_merged_df[['aromatic', 'heterocycle_present', 'peptide_like', 'lipid_like',
                        'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass', 'pains_flag']].dtypes)

print("\n✓ Missing value check:")
print(final_merged_df[['aromatic', 'heterocycle_present', 'peptide_like', 'lipid_like',
                        'lipinski_pass', 'veber_pass', 'egan_pass', 'ghose_pass', 'pains_flag']].isnull().sum())

✓ Sample of merged data:


,id,name,aromatic,heterocycle_present,lipinski_pass,veber_pass,egan_pass,ghose_pass,pains_flag
0,0,sulphasalazine,True,True,True,NaN,NaN,False,False
1,1,moxalactam,True,True,False,False,False,False,True
2,2,clioquinol,True,True,True,True,True,False,False
3,3,bbcpd11 (cimetidine analog) (y-g13),True,True,True,NaN,NaN,True,True
4,4,schembl614298,True,True,True,NaN,NaN,False,True
5,5,"uk-240,455",True,True,True,False,False,True,False
6,6,morphine-6-glucuronide,True,True,True,NaN,NaN,False,True
7,7,nitrofurantoin,True,True,True,NaN,NaN,False,False
8,8,"l-701,324",True,True,False,True,False,False,False
9,9,33419-42-0,True,True,False,NaN,NaN,False,True



✓ Data type check:
aromatic               object
heterocycle_present      bool
peptide_like           object
lipid_like             object
lipinski_pass            bool
veber_pass             object
egan_pass              object
ghose_pass             object
pains_flag             object
dtype: object

✓ Missing value check:
aromatic                 13
heterocycle_present       0
peptide_like             13
lipid_like               13
lipinski_pass             0
veber_pass             2586
egan_pass              2586
ghose_pass               64
pains_flag               13
dtype: int64
